# 🧠 RSNA Knee Multimodal AI - Phase 4: Multi-View Vision Model Training

This notebook demonstrates the end-to-end deep learning vision architecture, loss formulations, slice attention interpretability, and cross-validation training pipeline for multi-label knee pathology detection across **12 target abnormalities**.

### Pipeline Highlights:
1. **Multi-View Tri-Plane Deep Learning Model (`MultiViewKneeModel`)**: Dedicated 2.5D feature encoders for `Sagittal`, `Coronal`, and `Axial` sequences.
2. **Pathology-Aware Slice Attention Pooling**: Learns diagnostic slice importance distributions $\alpha \in \mathbb{R}^D$ per plane.
3. **Cross-Plane Gated Fusion**: Fuses orthogonal anatomical planes using gated residual projections.
4. **Unified Soft-Target Multi-Label Loss (`SoftBCEWithLogitsLoss`)**: Trains jointly on continuous NLP pseudo-labels ($N=4,407$) and upweighted gold-standard annotations ($N=58$).
5. **Validation Engine**: Automated Macro-AUC optimization, per-pathology metric tracking, and checkpoint saving.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is accessible
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

from src.models import MultiViewKneeModel, SingleViewKneeModel, TARGET_COLS, PLANES
from src.losses import SoftBCEWithLogitsLoss, AsymmetricFocalLoss, CombinedKneeLoss
from src.trainer import ModelTrainer, calculate_metrics, evaluate_model
from src.dataset import create_dataloaders

sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
print(f"Target Pathologies ({len(TARGET_COLS)}): {', '.join(TARGET_COLS)}")

## 1. 🏗️ Model Instantiation & Parameter Architecture
Let's initialize the `MultiViewKneeModel` and inspect the parameter distribution across the encoders, slice attention poolers, fusion layer, and classification head.

In [ ]:
model = MultiViewKneeModel(
    num_classes=12,
    feature_dim=128,
    fused_dim=256,
    dropout=0.25
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=== MultiViewKneeModel Architecture Summary ===")
print(f"• Total Parameters:     {total_params:,}")
print(f"• Trainable Parameters: {trainable_params:,}")
print(f"• Encoders:             Sagittal, Coronal, Axial (2.5D ConvNeXt-style Stem + 3 Stages)")
print(f"• Slice Aggregation:    SliceAttentionPooling (Learned diagnostic softmax weights)")
print(f"• Fusion Layer:         CrossPlaneFusion (Gated projection {128*3} -> 256)")
print(f"• Classifier Head:      256 -> 128 -> 12 Multi-Label Logits")

## 2. 🔍 Diagnostic Slice Attention Interpretability
In knee MRI, critical pathologies are concentrated on specific anatomical slices. The attention pooling mechanism learns which slices are diagnostically relevant for each study.

In [ ]:
# Simulate a multi-view input study: [Batch=1, Planes=3, Depth=16, H=224, W=224]
sample_input = torch.randn(1, 3, 16, 224, 224).to(device)
logits, attn_weights = model(sample_input, return_attention=True)

fig, ax = plt.subplots(figsize=(12, 4))
slices = np.arange(1, 17)
width = 0.25

sag_attn = attn_weights['sagittal'].detach().cpu().numpy()[0]
cor_attn = attn_weights['coronal'].detach().cpu().numpy()[0]
ax_attn  = attn_weights['axial'].detach().cpu().numpy()[0]

ax.bar(slices - width, sag_attn, width=width, label='Sagittal (e.g. ACL, Meniscus)', color='#1f77b4')
ax.bar(slices, cor_attn, width=width, label='Coronal (e.g. MCL, Meniscus Root)', color='#ff7f0e')
ax.bar(slices + width, ax_attn, width=width, label='Axial (e.g. Patella, Effusion)', color='#2ca02c')

ax.set_title('Learned Slice Attention Distribution Across 16 Equidistant Slices', fontsize=12, fontweight='bold')
ax.set_xlabel('Slice Index along Acquisition Trajectory (1 -> 16)', fontsize=10)
ax.set_ylabel('Attention Weight (Softmax Normalized)', fontsize=10)
ax.set_xticks(slices)
ax.legend()
plt.tight_layout()
plt.show()

## 3. 🎯 Multi-Label Soft-Target Loss Formulations
Demonstrating loss behavior on continuous pseudo-probabilities versus hard binary gold ground truth.

In [ ]:
soft_bce = SoftBCEWithLogitsLoss(gold_weight=2.0)
asym_focal = AsymmetricFocalLoss(gamma_neg=4.0, gamma_pos=1.0)
combined_loss = CombinedKneeLoss(bce_weight=0.7, focal_weight=0.3, gold_weight=2.0)

# Sample targets: 1 Gold Study, 1 Pseudo-Labeled Study
sample_targets = torch.tensor([
    [1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0],  # Gold study
    [0.92, 0.05, 0.85, 0.05, 0.75, 0.80, 0.05, 0.88, 0.05, 0.65, 0.05, 0.05]  # NLP pseudo-probs
]).to(device)
has_gold_flags = torch.tensor([1.0, 0.0]).to(device)

sample_logits = torch.randn(2, 12).to(device)

loss_bce_val = soft_bce(sample_logits, sample_targets, has_gold_flags).item()
loss_focal_val = asym_focal(sample_logits, sample_targets, has_gold_flags).item()
loss_comb_val = combined_loss(sample_logits, sample_targets, has_gold_flags).item()

print("=== Loss Comparisons ===")
print(f"• Soft BCE Loss:         {loss_bce_val:.4f}")
print(f"• Asymmetric Focal Loss: {loss_focal_val:.4f}")
print(f"• Combined Hybrid Loss:  {loss_comb_val:.4f}")

## 4. ⚡ Cross-Validation Training & Model Evaluation
Let's instantiate the training engine for Fold 0 and evaluate predictions across all 12 target pathologies.

In [ ]:
train_loader, val_loader = create_dataloaders(
    df_path='../data/train_folds.csv',
    series_path='../data/train_series.csv',
    val_fold=0,
    batch_size=8,
    num_workers=0,
    target_depth=16,
    target_spatial=(224, 224),
    mode='pseudo_labeled'
)

trainer = ModelTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=CombinedKneeLoss(gold_weight=2.0),
    device=device,
    save_dir='../models',
    fold_id=0
)

# Evaluate untrained initial baseline
eval_df, initial_macro_auc = evaluate_model(model, val_loader, device=device)
print(f"Initial Zero-Shot Baseline Macro-AUC: {initial_macro_auc:.4f}\n")
print("=== Pathology Performance Breakdown ===")
print(eval_df.to_string(index=False))